# 1.3. Manuales → Base de Datos de Grafos (Neo4j)
#### Objetivo: Extraer productos, componentes y sus relaciones de los manuales, y crear un grafo de conocimiento.

## Pasos sugeridos
1. Leer todos los archivos Markdown de manuales_productos/.
2. Usar un LLM para extraer entidades y relaciones.
3. Crear nodos y relaciones en Neo4j.

## Nodos a crear

- Producto: {id, nombre, categoria, marca}
- Componente: {nombre, tipo, especificacion}
- Procedimiento: {nombre, descripcion}

## Relaciones a crear

- (Producto)-[:TIENE_COMPONENTE]->(Componente)
- (Producto)-[:COMPATIBLE_CON]->(Producto)
- (Producto)-[:COMPARTE_REPUESTO {tipo}]->(Producto)
- (Componente)-[:USADO_EN]->(Producto)
- (Producto)-[:TIENE_PROCEDIMIENTO]->(Procedimiento)

## Ejemplo de extracción
- Texto: "El motor de 1200W es compartido con la Procesadora (P007)"

{​
"producto1": "P001",​
"producto2": "P007",​
"componente": "Motor 1200W",​
"relacion": "COMPARTE_COMPONENTE"​
}

## Librerias

In [1]:
import json
import os
import configparser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
config = configparser.ConfigParser()
config.read('config.ini')


['config.ini']

## Conectar con LLM

In [2]:
GOOGLE_API_KEY = config['GOOGLE']['API_KEY']

MODELO = 'gemma-4-31b-it' # 'gemini-2.5-flash-lite' #  # 'gemini-2.5-flash' es más rápido pero menos potente que 'gemma-4-31b-it'
llm = ChatGoogleGenerativeAI(
    model=MODELO,
    temperature=0.0,
    thinking_level="minimal",   # Configura el nivel de pensamiento al mínimo
    api_key=GOOGLE_API_KEY
)

def extraer_texto(content):
    """Extrae el texto limpio de la respuesta del modelo"""
    if isinstance(content, str):
        try:
            # Intenta parsear como JSON si es string
            bloques = json.loads(content)
            if isinstance(bloques, list):
                return next((item['text'] for item in bloques if item.get('type') == 'text'), content)
        except:
            pass
    elif isinstance(content, list):
        # Si es lista directamente
        return next((item['text'] for item in content if item.get('type') == 'text'), '')
    
    return str(content)


## Leer manuales y procesarlos

In [3]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = """
Actúa como un ingeniero de datos experto en la creación de Grafos de Conocimiento (Knowledge Graphs). Tu tarea es leer un manual técnico de productos y extraer información estructurada estrictamente en formato JSON, identificando Nodos (entidades) y Relaciones, con sus respectivas propiedades.
"""

prompt_extraction = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("user", """
### REGLAS DE EXTRACCIÓN DE NODOS
Debes identificar y extraer los siguientes tipos de nodos con sus propiedades exactas. Si una propiedad no se menciona explícitamente pero se puede deducir del contexto (como la marca al final del documento), hazlo. Si es imposible deducirla, usa "null".

1. Producto
   - id: El código alfanumérico (ej. "P007").
   - nombre: Nombre del producto (ej. "Procesadora").
   - categoria: Categoría general (ej. "Batidoras y Procesadoras Profesionales").
   - marca: La marca a la que pertenece (ej. "TechHome").

2. Componente
   - nombre: Nombre del componente (ej. "Motor de Inducción", "Disco Rallador Grueso").
   - tipo: Clasificación del componente (ej. "Motor", "Accesorio", "Placa").
   - especificacion: Dato técnico relevante (ej. "800W", "Acero Inoxidable 304").

3. Procedimiento
   - nombre: Nombre de la acción (ej. "Rallar Queso").
   - descripcion: Breve resumen de los pasos o propósito del procedimiento.

### REGLAS DE EXTRACCIÓN DE RELACIONES
Debes identificar cómo se conectan los nodos extraídos utilizando ÚNICAMENTE los siguientes tipos de relaciones:

1. (Producto)-[:TIENE_COMPONENTE]->(Componente)
2. (Producto)-[:COMPATIBLE_CON]->(Producto)
3. (Producto)-[:COMPARTE_REPUESTO]->(Producto)
4. (Componente)-[:USADO_EN]->(Producto)
5. (Producto)-[:TIENE_PROCEDIMIENTO]->(Procedimiento)

### FORMATO DE SALIDA (ESTRICTO)
Tu respuesta debe ser ÚNICAMENTE un objeto JSON válido. No incluyas texto antes ni después del JSON. No uses bloques de código markdown (```json) en la salida final, solo el texto JSON puro. 

Estructura el JSON de la siguiente manera:

{{
  "nodos": {{
    "productos": [
      {{"id": "P007", "nombre": "Procesadora", "categoria": "Procesadoras Profesionales", "marca": "TechHome"}}
    ],
    "componentes": [
      {{"nombre": "Motor 800W", "tipo": "Motor", "especificacion": "800W continuos, 1200W pico"}}
    ],
    "procedimientos": [
      {{"nombre": "Rallar Queso", "descripcion": "Instalación de disco grueso y uso de velocidad 6 para procesar queso"}}
    ]
  }},
  "relaciones": [
    {{"Producto": "P007", "Producto": "P001", "relacion": "COMPARTE_REPUESTO"}},
    {{"Producto": "P007", "Componente": "Motor 800W", "relacion": "TIENE_COMPONENTE"}},
    {{"Producto": "Motor 800W", "Producto": "P007", "relacion": "USADO_EN"}},
    {{"Producto": "P007", "Procedimiento": "Rallar Queso", "relacion": "TIENE_PROCEDIMIENTO"}}
  ]
}}

### TEXTO A ANALIZAR:
{document}""")
])

# Ejecución de la cadena
chain_extraction = prompt_extraction | llm

## Conectar con Neo4j

In [26]:
# Conectar a Neo4j y cargar los datos extraídos
from neo4j import GraphDatabase
class Neo4jHandler:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def crear_nodo_producto(self, id, nombre, categoria, marca):
        with self.driver.session() as session:
            session.run(
                "MERGE (p:Producto {id: $id}) "
                "SET p.nombre = $nombre, p.categoria = $categoria, p.marca = $marca",
                id=id, nombre=nombre, categoria=categoria, marca=marca
            )

    def crear_nodo_componente(self, nombre, tipo, especificacion):
        with self.driver.session() as session:
            session.run(
                "MERGE (c:Componente {nombre: $nombre}) "
                "SET c.tipo = $tipo, c.especificacion = $especificacion",
                nombre=nombre, tipo=tipo, especificacion=especificacion
            )

    def crear_nodo_procedimiento(self, nombre, descripcion):
        with self.driver.session() as session:
            session.run(
                "MERGE (pr:Procedimiento {nombre: $nombre}) "
                "SET pr.descripcion = $descripcion",
                nombre=nombre, descripcion=descripcion
            )

    def crear_relacion(self, nodo1_label, nodo1_prop, nodo1_valor,
                        nodo2_label, nodo2_prop, nodo2_valor,
                        relacion_tipo, propiedades={}):
        with self.driver.session() as session:
            props_str = ", ".join([f"{k}: ${k}" for k in propiedades.keys()])
            query = (
                f"MATCH (a:{nodo1_label} {{{nodo1_prop}: $nodo1_valor}}), "
                f"(b:{nodo2_label} {{{nodo2_prop}: $nodo2_valor}}) "
                f"MERGE (a)-[r:{relacion_tipo} {{ {props_str} }}]->(b)"
            )
            params = {"nodo1_valor": nodo1_valor, "nodo2_valor": nodo2_valor}
            params.update(propiedades)
            session.run(query, **params)

    # Funcion para buscar nodos o relaciones
    def buscar_nodo(self, label, prop, valor):
        with self.driver.session() as session:
            result = session.run(f"MATCH (n:{label} {{{prop}: $valor}}) RETURN n", valor=valor)
            return [record["n"] for record in result]   

    # Función para limpiar la base de datos
    def clear_database(self):
        with self.driver.session() as session:
            session.run("MATCH (n) DETACH DELETE n")
    

In [ ]:
# Configuración de conexión a Neo4j
# Parametros de config.ini
NEO4J_URI = config.get("NEO4J", "NEO4J_URI")
NEO4J_USER = config.get("NEO4J", "NEO4J_USER")
NEO4J_PASSWORD = config.get("NEO4J", "NEO4J_PASSWORD")
neo4j_handler = Neo4jHandler(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

#neo4j_handler.clear_database()  # Limpia la base de datos antes de cargar nuevos datos

# Cierra la conexión al finalizar
#neo4j_handler.close()


In [31]:
# Recorrer la variable respuesta_LLM y usar el handler para cargar los datos en Neo4j

def cargar_datos_en_neo4j(respuesta_LLM, neo4j_handler):
  """{
    "nodos": {
      "productos": [
        {
          "id": "P010",
          "nombre": "Freidora de Aire",
          "categoria": "Cocción Saludable sin Aceite",
          "marca": "HealthyCook / ToastMaster"
        },"""
  # convertir la respuesta del LLM a un diccionario de Python
  respuesta_LLM_json = json.loads(respuesta_LLM)

  for producto in respuesta_LLM_json.get("nodos", {}).get("productos", []):
    # Chequear que el producto ya no exista en la base de datos antes de crearlo
    resultado = neo4j_handler.buscar_nodo("Producto", "id", producto.get("id", "null"))
    if not resultado:  # Si no existe, crear el nodo
      neo4j_handler.crear_nodo_producto(
        id=producto.get("id", "null"),
        nombre=producto.get("nombre", "null"),
        categoria=producto.get("categoria", "null"),
        marca=producto.get("marca", "null")
      )

  """
  "componentes": [
        {
          "nombre": "Resistencia de Calentamiento",
          "tipo": "Resistencia",
          "especificacion": "1400W, Acero inoxidable con recubrimiento cerámico"
        },
        {
          "nombre": "Ventilador de Alta Velocidad",
          "tipo": "Motor",
          "especificacion": "60W, 3200 RPM, 120 CFM"
        },
  """

  for componente in respuesta_LLM_json.get("nodos", {}).get("componentes", []):
    # Chequear que el componente ya no exista en la base de datos antes de crearlo
    resultado = neo4j_handler.buscar_nodo("Componente", "nombre", componente.get("nombre", "null"))
    if not resultado:  # Si no existe, crear el nodo
      neo4j_handler.crear_nodo_componente(
        nombre=componente.get("nombre", "null"),
        tipo=componente.get("tipo", "null"),
        especificacion=componente.get("especificacion", "null")
      )

  """ 
  "procedimientos": [
        {
          "nombre": "Cocción de Papas Fritas",
          "descripcion": "Remojar 30min, secar, rociar aceite, cocinar 12min a 180°C, sacudir, 8min a 200°C"
        },
      """

  for procedimiento in respuesta_LLM_json.get("nodos", {}).get("procedimientos", []):
    # Chequear que el procedimiento ya no exista en la base de datos antes de crearlo
    resultado = neo4j_handler.buscar_nodo("Procedimiento", "nombre", procedimiento.get("nombre", "null"))
    if not resultado:  # Si no existe, crear el nodo
      neo4j_handler.crear_nodo_procedimiento(
        nombre=procedimiento.get("nombre", "null"),
        descripcion=procedimiento.get("descripcion", "null")
      )

  """
  "relaciones": [
      {
        "Producto": "P010",
        "Componente": "Resistencia de Calentamiento",
        "relacion": "TIENE_COMPONENTE"
      },
  """
  for relacion in respuesta_LLM_json.get("relaciones", []):
    tipo_relacion = relacion.get("relacion", "null")
    if tipo_relacion in ["TIENE_COMPONENTE", "COMPATIBLE_CON", "COMPARTE_REPUESTO", "USADO_EN", "TIENE_PROCEDIMIENTO"]:
      # Determinar los nodos involucrados
      nodo1_label, nodo1_valor = None, None
      nodo2_label, nodo2_valor = None, None
      
      for key in relacion.keys():
        if key == "Producto":
          if not nodo1_label:
            nodo1_label, nodo1_valor = "Producto", relacion[key]
          else:
            nodo2_label, nodo2_valor = "Producto", relacion[key]
        elif key == "Componente":
          nodo2_label, nodo2_valor = "Componente", relacion[key]
        elif key == "Procedimiento":
          nodo2_label, nodo2_valor = "Procedimiento", relacion[key]

      if nodo1_label and nodo2_label:
        neo4j_handler.crear_relacion(
          nodo1_label=nodo1_label,
          nodo1_prop="id" if nodo1_label == "Producto" else "nombre",
          nodo1_valor=nodo1_valor,
          nodo2_label=nodo2_label,
          nodo2_prop="id" if nodo2_label == "Producto" else "nombre",
          nodo2_valor=nodo2_valor,
          relacion_tipo=tipo_relacion
        )

In [33]:
def procesar_manual_y_cargar_en_neo4j(filename):
    with open(os.path.join("data/manuales_productos/", filename), "r") as file:
        contenido = file.read()
        print(f"Procesando el archivo {filename}:")
        # Asumiendo que 'contenido' es tu variable con el manual
        response_extraction = chain_extraction.invoke({"document": contenido})

        print("=== INFORMACIÓN EXTRAÍDA ===\n")
        # Asegúrate de tener definida tu función 'extraer_texto'
        respuesta_LLM = extraer_texto(response_extraction.content)
        print(respuesta_LLM)
        cargar_datos_en_neo4j(respuesta_LLM, neo4j_handler)
        print(f"Datos del archivo {filename} cargados en Neo4j.\n\n")
        

In [35]:
# manual_batidoras_profesionales.md manual_bebidas_calientes.md  manual_climatizacion_hogar.md  manual_coccion_mayor.md  manual_coccion_saludable.md  manual_licuadoras.md  manual_limpieza_inteligente.md

# procesar_manual_y_cargar_en_neo4j("manual_batidoras_profesionales.md")
# procesar_manual_y_cargar_en_neo4j("manual_bebidas_calientes.md")
procesar_manual_y_cargar_en_neo4j("manual_climatizacion_hogar.md")
procesar_manual_y_cargar_en_neo4j("manual_coccion_mayor.md")
procesar_manual_y_cargar_en_neo4j("manual_coccion_saludable.md")
procesar_manual_y_cargar_en_neo4j("manual_licuadoras.md")
procesar_manual_y_cargar_en_neo4j("manual_limpieza_inteligente.md")



Procesando el archivo manual_climatizacion_hogar.md:
=== INFORMACIÓN EXTRAÍDA ===

{
  "nodos": {
    "productos": [
      {
        "id": "P019",
        "nombre": "Ventilador de Torre",
        "categoria": "Climatización y Cuidado del Hogar",
        "marca": "AirFlow"
      },
      {
        "id": "P020",
        "nombre": "Purificador de Aire",
        "categoria": "Climatización y Cuidado del Hogar",
        "marca": "AirFlow"
      },
      {
        "id": "P018",
        "nombre": "Plancha a Vapor",
        "categoria": "Climatización y Cuidado del Hogar",
        "marca": "IronMaster"
      }
    ],
    "componentes": [
      {
        "nombre": "Motor de Ventilación",
        "tipo": "Motor",
        "especificacion": "Motor AC asíncrono, 80W"
      },
      {
        "nombre": "Turbina de Flujo Cruzado",
        "tipo": "Turbina",
        "especificacion": "Plástico ABS balanceado, 80cm"
      },
      {
        "nombre": "Sistema de Oscilación",
        "tipo": "Motor",
  